# Microsoft Recommenders NRMS / MIND PyTorch 版

这份 notebook 对应 Microsoft Recommenders 的 `examples/00_quick_start/nrms_MIND.ipynb`，但这里完全改成 PyTorch，不使用 TensorFlow，也不依赖 Microsoft `recommenders` 包。

目标是把 NRMS 的核心结构讲清楚并跑通：

- MIND 新闻推荐数据格式：`news.tsv` 和 `behaviors.tsv` 的核心字段。
- 新闻编码器：词向量 + 多头自注意力 + 注意力池化。
- 用户编码器：历史点击新闻序列 + 多头自注意力 + 注意力池化。
- 点击预测：用户向量和候选新闻向量做点积，用二分类损失训练。


## 1. 导入依赖并固定随机种子

整份 notebook 只使用 PyTorch 相关深度学习组件。Pandas 只负责组织表格数据，Numpy 只用于指标计算。

In [ ]:
from collections import Counter
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前训练设备：{device}")


## 2. 构造一个迷你 MIND 数据集

真实 MIND 数据里，`news.tsv` 描述新闻内容，`behaviors.tsv` 描述用户历史点击和本次曝光候选。为了让 notebook 稳定、快速、离线可跑，这里直接构造一个很小的 MIND 风格样例。

In [ ]:
news_df = pd.DataFrame([
    ["N1", "sports", "nba finals preview warriors celtics"],
    ["N2", "sports", "football transfer window latest news"],
    ["N3", "finance", "stock market rallies after inflation report"],
    ["N4", "finance", "federal reserve signals rate decision"],
    ["N5", "tech", "new ai chip improves training speed"],
    ["N6", "tech", "smartphone camera review and benchmark"],
    ["N7", "travel", "city guide for weekend family trip"],
    ["N8", "health", "sleep habits improve mental health"],
    ["N9", "world", "global leaders meet for climate summit"],
    ["N10", "entertainment", "movie awards highlight new directors"],
], columns=["news_id", "category", "title"])

behaviors_df = pd.DataFrame([
    ["I1", "U1", "N1 N2 N5", "N3-0 N5-1 N7-0 N6-0"],
    ["I2", "U2", "N3 N4 N9", "N4-1 N1-0 N5-0 N8-0"],
    ["I3", "U3", "N7 N8 N10", "N8-1 N2-0 N6-0 N9-0"],
    ["I4", "U4", "N5 N6 N3", "N6-1 N10-0 N2-0 N4-0"],
    ["I5", "U1", "N1 N5 N6", "N1-1 N4-0 N8-0 N9-0"],
    ["I6", "U2", "N3 N4 N5", "N3-1 N7-0 N10-0 N2-0"],
], columns=["impression_id", "user_id", "history", "impressions"])

display(news_df)
display(behaviors_df)


## 3. 文本编码

NRMS 的新闻编码器输入是标题词序列。这里用空格分词，构造词表，并把每条新闻标题填充或截断到固定长度。真实任务中可以替换为更完整的分词、预训练词向量或中文 tokenizer。

In [ ]:
max_title_len = 8
max_history_len = 4
min_word_count = 1

def tokenize(text):
    return text.lower().split()

word_counter = Counter()
for title in news_df["title"]:
    word_counter.update(tokenize(title))

word_to_id = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counter.items():
    if count >= min_word_count:
        word_to_id[word] = len(word_to_id)

def encode_title(title):
    ids = [word_to_id.get(word, word_to_id["<UNK>"]) for word in tokenize(title)]
    ids = ids[:max_title_len]
    ids = ids + [word_to_id["<PAD>"]] * (max_title_len - len(ids))
    return ids

news_id_to_title_ids = {row.news_id: encode_title(row.title) for row in news_df.itertuples()}
zero_title = [word_to_id["<PAD>"]] * max_title_len

print(f"词表大小：{len(word_to_id)}")
print("N5 标题编码：", news_id_to_title_ids["N5"])


## 4. 把曝光日志展开成训练样本

MIND 的 `impressions` 字段通常长这样：`N3-0 N5-1 N7-0`，其中 `1` 表示点击，`0` 表示未点击。我们把每个候选新闻展开成一条二分类样本。

In [ ]:
def encode_history(history_text):
    history_ids = history_text.split()[-max_history_len:]
    encoded = [news_id_to_title_ids.get(news_id, zero_title) for news_id in history_ids]
    if len(encoded) < max_history_len:
        encoded = [zero_title] * (max_history_len - len(encoded)) + encoded
    return encoded

records = []
for row in behaviors_df.itertuples(index=False):
    history = encode_history(row.history)
    for item in row.impressions.split():
        news_id, label = item.split("-")
        records.append({
            "impression_id": row.impression_id,
            "user_id": row.user_id,
            "history": history,
            "candidate": news_id_to_title_ids[news_id],
            "label": float(label),
            "candidate_news_id": news_id,
        })

samples_df = pd.DataFrame(records)
print(f"训练样本数：{len(samples_df)}")
display(samples_df[["impression_id", "user_id", "candidate_news_id", "label"]].head(8))


## 5. Dataset 与 DataLoader

每条样本包含三部分：用户历史点击新闻标题、候选新闻标题、点击标签。历史点击是一个二维矩阵：`历史新闻数 × 标题词数`。

In [ ]:
class MindToyDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples.reset_index(drop=True)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples.iloc[index]
        history = torch.tensor(row["history"], dtype=torch.long)
        candidate = torch.tensor(row["candidate"], dtype=torch.long)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return history, candidate, label, row["impression_id"]

dataset = MindToyDataset(samples_df)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

for history, candidate, label, impression_id in loader:
    print("history shape:", history.shape)
    print("candidate shape:", candidate.shape)
    print("label shape:", label.shape)
    break


## 6. NRMS 模型结构

NRMS 的核心是两层编码器：

- 新闻编码器：先把标题词 ID 变成词向量，再用多头自注意力建模标题内部词之间的关系，最后用注意力池化得到新闻向量。
- 用户编码器：把历史点击新闻逐条编码成新闻向量，再用多头自注意力建模用户兴趣之间的关系，最后用注意力池化得到用户向量。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, input_dim, attention_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, attention_dim)
        self.query = nn.Parameter(torch.empty(attention_dim))
        nn.init.normal_(self.query, mean=0.0, std=0.1)

    def forward(self, x, mask):
        scores = torch.matmul(torch.tanh(self.proj(x)), self.query)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)


class NewsEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=4, attention_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.additive_attention = AdditiveAttention(embed_dim, attention_dim)

    def forward(self, title_ids):
        mask = title_ids != 0
        embedded = self.embedding(title_ids)
        attended, _ = self.self_attention(embedded, embedded, embedded, key_padding_mask=~mask)
        return self.additive_attention(attended, mask)


class UserEncoder(nn.Module):
    def __init__(self, news_encoder, embed_dim=32, num_heads=4, attention_dim=32):
        super().__init__()
        self.news_encoder = news_encoder
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.additive_attention = AdditiveAttention(embed_dim, attention_dim)

    def forward(self, history_title_ids):
        batch_size, history_len, title_len = history_title_ids.shape
        flat_history = history_title_ids.reshape(batch_size * history_len, title_len)
        news_vectors = self.news_encoder(flat_history).reshape(batch_size, history_len, -1)
        history_mask = history_title_ids.sum(dim=-1) != 0
        attended, _ = self.self_attention(news_vectors, news_vectors, news_vectors, key_padding_mask=~history_mask)
        return self.additive_attention(attended, history_mask)


class NRMS(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=4, attention_dim=32):
        super().__init__()
        self.news_encoder = NewsEncoder(vocab_size, embed_dim, num_heads, attention_dim)
        self.user_encoder = UserEncoder(self.news_encoder, embed_dim, num_heads, attention_dim)

    def forward(self, history_title_ids, candidate_title_ids):
        user_vector = self.user_encoder(history_title_ids)
        candidate_vector = self.news_encoder(candidate_title_ids)
        return torch.sum(user_vector * candidate_vector, dim=1)

model = NRMS(vocab_size=len(word_to_id)).to(device)
print(model)


## 7. 训练一个可跑通的小模型

这个迷你数据集只用于讲清楚流程，不能代表真实 MIND 的效果。真实训练时通常需要更大的 MIND small/large 数据、负采样、更长训练轮数和验证集。

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

for epoch in range(8):
    model.train()
    total_loss = 0.0
    total_count = 0
    for history, candidate, label, _ in loader:
        history = history.to(device)
        candidate = candidate.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        logits = model(history, candidate)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(label)
        total_count += len(label)

    print(f"第 {epoch + 1} 轮，平均损失：{total_loss / total_count:.4f}")


## 8. 新闻推荐常用指标

MIND/NRMS 常见评估指标包括 Group AUC、MRR、NDCG@5、NDCG@10。它们都按一次曝光里的候选列表来算，再对所有曝光求平均。

In [ ]:
def mrr_score(labels, scores):
    order = np.argsort(scores)[::-1]
    ranked_labels = np.asarray(labels)[order]
    positive_positions = np.where(ranked_labels == 1)[0]
    return 0.0 if len(positive_positions) == 0 else 1.0 / (positive_positions[0] + 1)


def ndcg_score(labels, scores, k):
    order = np.argsort(scores)[::-1][:k]
    ranked_labels = np.asarray(labels)[order]
    discounts = 1 / np.log2(np.arange(2, len(ranked_labels) + 2))
    dcg = np.sum(ranked_labels * discounts)
    ideal_labels = np.sort(labels)[::-1][:k]
    ideal_dcg = np.sum(ideal_labels * discounts[:len(ideal_labels)])
    return float(dcg / ideal_dcg) if ideal_dcg > 0 else 0.0


def group_auc_score(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan
    pair_scores = (pos_scores[:, None] > neg_scores[None, :]).astype(float)
    pair_scores += 0.5 * (pos_scores[:, None] == neg_scores[None, :]).astype(float)
    return float(pair_scores.mean())


def evaluate_by_impression(result_df):
    rows = []
    for impression_id, group in result_df.groupby("impression_id"):
        labels = group["label"].to_numpy()
        scores = group["score"].to_numpy()
        rows.append({
            "impression_id": impression_id,
            "group_auc": group_auc_score(labels, scores),
            "mrr": mrr_score(labels, scores),
            "ndcg@5": ndcg_score(labels, scores, 5),
            "ndcg@10": ndcg_score(labels, scores, 10),
        })
    metric_df = pd.DataFrame(rows)
    return metric_df, metric_df.drop(columns=["impression_id"]).mean(numeric_only=True)

model.eval()
all_rows = []
full_loader = DataLoader(dataset, batch_size=8, shuffle=False)
with torch.no_grad():
    for history, candidate, label, impression_id in full_loader:
        logits = model(history.to(device), candidate.to(device))
        scores = torch.sigmoid(logits).cpu().numpy()
        for imp, y, score in zip(impression_id, label.numpy(), scores):
            all_rows.append({"impression_id": imp, "label": y, "score": score})

result_df = pd.DataFrame(all_rows)
metric_df, metrics = evaluate_by_impression(result_df)
display(metric_df)
print(metrics)


## 9. 保存 PyTorch 权重

这里保存的是教学样例的小模型。真实项目里可以把词表、新闻 ID 映射、模型参数和训练配置一起保存。

In [ ]:
artifact_dir = "../artifacts"
import os
os.makedirs(artifact_dir, exist_ok=True)
model_path = os.path.join(artifact_dir, "nrms_mind_torch_demo.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "word_to_id": word_to_id,
    "max_title_len": max_title_len,
    "max_history_len": max_history_len,
}, model_path)
print(f"模型已保存到：{model_path}")


## 10. 下一步怎么替换成真实 MIND

如果要接入真实 MIND 数据，可以保留本 notebook 的模型结构和指标函数，把第 2 到第 4 节替换为真实 `news.tsv`、`behaviors.tsv` 的读取逻辑。真实数据中标题通常需要更规范的分词和词向量初始化，训练时也建议使用按曝光分组的负采样。